# kNNDM spatial cross-validation lab

Explore prediction-oriented fold construction by comparing nearest-neighbour distances from held-out points with those from a prediction domain. This is a KMeans/Euclidean teaching implementation, not a complete CAST reproduction or the original NNDM leave-one-out method.

In [ ]:
from pathlib import Pathimport sys_root = Path.cwd()if not (_root / 'implementations').exists():    _root = _root.parentsys.path.insert(0, str(_root / 'implementations' / 'python' / 'knndm'))import numpy as np
import matplotlib.pyplot as plt
from knndm import knndm_cv

## Build clustered training data and a prediction domain

The prediction points span the full domain, while training observations are concentrated around four centres.

In [ ]:
rng = np.random.default_rng(7)
centres = np.array([[0, 0], [100, 0], [0, 100], [100, 100]])
training = np.vstack([c + rng.normal(0, 5, size=(20, 2)) for c in centres])
prediction = rng.uniform(-20, 120, size=(800, 2))
plt.figure(figsize=(7, 6))
plt.scatter(prediction[:, 0], prediction[:, 1], s=3, alpha=.15, label='prediction domain')
plt.scatter(training[:, 0], training[:, 1], s=18, label='training observations')
plt.legend(); plt.axis('equal'); plt.show()

## Select folds and inspect the distance match

A small Wasserstein W indicates that the held-out-to-training nearest-distance distribution more closely resembles the prediction-to-training distribution. It is a diagnostic, not a guarantee of unbiased map accuracy.

In [ ]:
result = knndm_cv(training, prediction, n_splits=5, candidate_count=25, random_state=7)
print('branch:', result['branch'])
print('selected q:', result['selected_q'])
print('Wasserstein W:', result['wasserstein_w'])
print('KS p-value:', result['ks_pvalue'])
print('fold sizes:', [len(f['test']) for f in result['folds']])

In [ ]:
plt.figure(figsize=(7, 4))
plt.hist(result['prediction_nearest_distances'], bins=30, density=True, alpha=.5, label='prediction to training')
plt.hist(result['cv_nearest_distances'], bins=30, density=True, alpha=.5, label='held-out to fold training')
plt.xlabel('Nearest-neighbour distance (coordinate units)'); plt.ylabel('Density'); plt.legend(); plt.show()

## Inspect the selected folds

Plot each fold's held-out observations. Use the resulting indices to fit and predict with a model, then calculate map-level metrics on the concatenated out-of-fold predictions rather than averaging fold metrics when fold sizes differ.

In [ ]:
plt.figure(figsize=(7, 6))
for fold in result['folds']:
    held = training[fold['test']]
    plt.scatter(held[:, 0], held[:, 1], s=20, label=f"fold {fold['fold']}")
plt.legend(); plt.axis('equal'); plt.show()

## Limitations

This lab uses projected Euclidean coordinates and KMeans clustering. It does not implement hierarchical clustering, spherical distances, exact CAST cluster merging, polygon sampling or the original NNDM LOO algorithm. Always verify the selected distance distributions and compare against independent probability-sample validation where possible.